In [4]:
from pathlib import Path
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
 
src = "../prepared_data.parquet"
out = Path("charts")
out.mkdir(parents=True, exist_ok=True)
 
sns.set_theme(style="whitegrid", context="notebook")
RED, GREY = "#E50914", "#564d4d"

In [5]:
def save(name):
    plt.tight_layout()
    plt.savefig(out / name, dpi=150)
    plt.close()

In [6]:
df = pd.read_parquet(src)
movies = df[df["type"] == "Movie"].copy()
shows = df[df["type"] == "TV Show"].copy()
movies["minutes"] = movies["duration"].str.extract(r"(\d+)")[0].astype(float)
shows["seasons"] = shows["duration"].str.extract(r"(\d+)")[0].astype(int)

In [7]:
counts = df["type"].value_counts()
print(counts, "\n")
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(counts, labels=counts.index, autopct="%1.1f%%", colors=[RED, GREY],
       startangle=90, wedgeprops=dict(width=0.45, edgecolor="white"))
ax.set_title("Content by type")
save("01_type_distribution.png")

type
Movie      5696
TV Show     219
Name: count, dtype: int64 



In [8]:
top_c = df["country"].value_counts().head(10)
print(top_c, "\n")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=top_c.values, y=top_c.index, color=RED, ax=ax)
ax.set(title="Top 10 countries by titles", xlabel="Titles", ylabel="")
for i, v in enumerate(top_c.values):
    ax.text(v + 15, i, str(v), va="center")
save("02_top_countries.png")

country
United States     2401
India              975
United Kingdom     406
Canada             189
France             156
Spain              140
Egypt              107
Nigeria            100
Japan               96
Mexico              93
Name: count, dtype: int64 



In [10]:
genres = df["genres"].explode().value_counts().head(10)
print(genres, "\n")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=genres.values, y=genres.index, color=GREY, ax=ax)
ax.set(title="Top 10 genres", xlabel="Titles", ylabel="")
save("03_top_genres.png")

genres
International Movies        2558
Dramas                      2330
Comedies                    1593
Action & Adventure           815
Documentaries                762
Independent Movies           742
Romantic Movies              588
Children & Family Movies     559
Thrillers                    558
Horror Movies                344
Name: count, dtype: int64 



In [11]:
ratings = df["rating"].value_counts()
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(x=ratings.index, y=ratings.values, color=RED, ax=ax)
ax.set(title="Titles by maturity rating", xlabel="", ylabel="Titles")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
save("04_ratings.png")

In [19]:
directors = (df["director"]
    .apply(lambda v: [x.strip() for x in (v.split(",") if isinstance(v, str) else v)])
    .explode()
    .loc[lambda s: s.notna() & (s != "") & (s.str.lower() != "unknown")]
    .value_counts()
    .head(10))
print(directors, "\n")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=directors.values, y=directors.index, color=GREY, ax=ax)
ax.set(title="Top 10 directors", xlabel="Titles", ylabel="")
save("05_top_directors.png")

director
Rajiv Chilaka          22
Jan Suter              21
Alastair Fothergill    19
Raúl Campos            19
Suhas Kadav            16
Marcus Raboy           16
Jay Karas              15
Cathy Garcia-Molina    13
Youssef Chahine        12
Martin Scorsese        12
Name: count, dtype: int64 



In [13]:
yearly = df.groupby(["year_added", "type"]).size().unstack(fill_value=0)
yearly = yearly.loc[yearly.index >= 2014]
fig, ax = plt.subplots(figsize=(9, 4.5))
yearly.plot(kind="bar", stacked=True, color=[RED, GREY], ax=ax)
ax.set(title="Titles added to Netflix per year", xlabel="Year added", ylabel="Titles")
plt.setp(ax.get_xticklabels(), rotation=0)
save("06_added_per_year.png")

In [14]:
fig, ax = plt.subplots(figsize=(9, 4.5))
rel = df[df["release_year"] >= 1990]["release_year"].value_counts().sort_index()
ax.plot(rel.index, rel.values, color=RED, lw=2)
ax.fill_between(rel.index, rel.values, color=RED, alpha=0.15)
ax.set(title="Titles by release year (1990+)", xlabel="Release year", ylabel="Titles")
save("07_release_year.png")

In [15]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(movies["minutes"], bins=40, color=RED, ax=ax)
ax.axvline(movies["minutes"].median(), color="black", ls="--",
           label=f"median {movies['minutes'].median():.0f} min")
ax.set(title="Movie runtime distribution", xlabel="Minutes")
ax.legend()
save("08_movie_runtime.png")

In [16]:
month = df["month_added"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=month.index, y=month.values, color=GREY, ax=ax)
ax.set(title="Titles added by month (all years)", xlabel="Month", ylabel="Titles")
save("09_month_added.png")

In [17]:
top5 = df["country"].value_counts().head(5).index
ct = (df[df["country"].isin(top5)].groupby(["country", "rating"]).size()
      .unstack(fill_value=0))
ct = ct.div(ct.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(ct, cmap="Reds", annot=True, fmt=".0%", cbar=False, ax=ax)
ax.set(title="Rating mix within the top 5 countries", xlabel="", ylabel="")
save("10_country_rating_heatmap.png")

In [21]:
movies.to_parquet("../movies.parquet", index=False)
shows.to_parquet("../shows.parquet", index=False)